In [3]:
import os
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph
from langchain_google_genai import ChatGoogleGenerativeAI

/Users/armaanjagirdar/Projects/ML_Project/env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.15) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [4]:
load_dotenv()

True

In [5]:
graph = Neo4jGraph(url="neo4j://127.0.0.1:7687", username=os.getenv("NEO4J_USER"), password=os.getenv("NEO4J_PASSWORD"), enhanced_schema=True)

In [6]:
graph.refresh_schema()
print(graph.schema)

Node properties:
- **Entity**
  - `name`: STRING Example: "Fruit lesions"
Relationship properties:

The relationships:
(:Entity)-[:AFFECTS]->(:Entity)
(:Entity)-[:PRESENTS]->(:Entity)
(:Entity)-[:TREATED_BY]->(:Entity)
(:Entity)-[:CAUSED_BY]->(:Entity)


In [7]:
from langchain_neo4j.chains.graph_qa.cypher import GraphCypherQAChain

llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm, 
    graph=graph, 
    verbose=True,
    allow_dangerous_requests=True,    
)

In [8]:
question = "Disease: Rice Brown Spot. How to cure it"
result = cypher_chain.invoke({"query": question})
print("Answer:", result['result'])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (d:Entity {name: "Rice Brown Spot"})-[:TREATED_BY]->(cure:Entity) RETURN cure.name
Full Context:
[{'cure.name': 'Apply balanced fertilization'}, {'cure.name': 'Dispose of infected rice straw'}, {'cure.name': 'Remove weeds'}, {'cure.name': 'Strengthen quarantine measures'}, {'cure.name': 'Use shallow water irrigation'}, {'cure.name': 'Balanced fertilization'}, {'cure.name': 'Shallow water irrigation'}]

> Finished chain.
Answer: To cure Rice Brown Spot, you can apply balanced fertilization, dispose of infected rice straw, remove weeds, strengthen quarantine measures, and use shallow water irrigation.
